# Figure 5 — Q4: does removing the batch term make the downstream answers better?

Differential abundance and cross-study prediction are what correction is for. This figure checks whether 'removes batch well' and 'answers the question well' are the same ranking, and ends with what to actually use.

Each panel has its own cell and renders on its own, sized to suit it alone. The last cell assembles all of them at the submission size (180 x 240 mm) and writes the PDF.

The panel functions live in `figures/fig5.R` and are **sourced, not copied** — edit them there and re-run the cell, so the notebook and `snakemake` produce the same figure.

## Setup

In [ ]:
# Panels come from the script, so the notebook and the submitted figure cannot drift apart.
# main() is guarded, so sourcing runs nothing.
if (basename(getwd()) == "figures") setwd("..")
stopifnot("run this from benchmark/ or benchmark/figures/" = file.exists("config/methods.tsv"))
source("figures/fig5.R")

results_dir <- "results"

## Data

In [ ]:
scored <- read_scores(file.path(results_dir, "scores.tsv"))
loso <- read.delim(file.path(results_dir, "loso.tsv"))
simulated <- scored[scored$regime == "sim" & scored$status == "ok", ]
confounding <- dataset_confounding(results_dir, unique(scored$dataset[scored$regime == "real"]))
names(confounding) <- sub("^real/", "", unique(scored$dataset[scored$regime == "real"]))

cat(nrow(loso), "leave-one-study-out folds,", sum(loso$status == "ok"), "scored\n")

## a. Differential abundance against two truths

Orange is what the simulation planted; green is what the same test finds on the batch-free table. The second is reported because planting fold changes renormalises the community, so untouched taxa genuinely shift.

In [ ]:
show_panel(panel_da(simulated[simulated$da_prop > 0, ]), 180, 82)

## b. With nothing planted

Null scenarios: the fraction of taxa each method still calls.

In [ ]:
show_panel(panel_null_da(simulated[simulated$da_prop == 0 & simulated$method != "oracle", ]), 100, 78)

## c. Cross-study prediction

Leave-one-study-out auROC. An **upper bound**: correction ran once on the whole dataset and saw the held-out labels. `raw` has no leakage and is the honest reference.

In [ ]:
show_panel(panel_loso(loso), 100, 78)

## d. Do the three criteria agree on an order?

The direct test of the contradiction in the literature: rank by batch removal, by DA recovery, by cross-study auROC.

In [ ]:
show_panel(panel_consistency(ranking_table(scored, loso)), 92, 82)

## e. What it costs to run

Median seconds against median peak memory; point area is the failure rate. Note the three orders of magnitude on each axis.

In [ ]:
show_panel(panel_cost(scored), 112, 82)

## f. Top three by goal and by design

The recommendation table: what to use, split by research goal and by how confounded the design is.

In [ ]:
show_panel(panel_recommendation(scored, loso, confounding), 170, 68)

## Assembled figure

Everything above, in the layout that goes to the journal: 180 x 240 mm, one page.
The panels shrink here — that is the real submission size, and the font sizes above were
chosen for it.

In [ ]:
figure <- (panel_da(simulated[simulated$da_prop > 0, ]) /
           (panel_null_da(simulated[simulated$da_prop == 0 & simulated$method != "oracle", ]) |
            panel_loso(loso)) /
           ((panel_consistency(ranking_table(scored, loso)) | panel_cost(scored)) +
            plot_layout(widths = c(1, 1.3))) /
           panel_recommendation(scored, loso, confounding)) +
  plot_layout(heights = c(1.3, 1.2, 1.2, 1)) +
  plot_annotation(tag_levels = "a") &
  theme(plot.tag = element_text(size = 9, face = "bold"))

save_figure(figure, "results/figures/fig5.pdf", 180, 240)
show_panel(figure, 180, 240, scale = 1.5)